[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# Tipos de variables y su `dtype`.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo I: *Datos y su calidad*

> 🔑 **Versión con soluciones.** Solo para el instructor o para revisión posterior.

En este *notebook* se clasificarán las variables por su tipo y su escala de medición, y se elegirá el `dtype` de *pandas* 3 que corresponde a cada una, junto con las consecuencias de elegir mal.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencia de la API: curso py311.

Este curso no repasa la API método por método: esa explicación detallada está en los *notebooks* del curso [py311](https://github.com/PythonistaMX/py311). Aquí se hace un repaso breve y el tiempo se dedica a las **decisiones de análisis**: qué tipo de dato elegir, qué hacer con un faltante y qué consecuencias tiene cada decisión. Para este *notebook*:

| Tema | *Notebooks* de py311 |
|---|---|
| *NumPy* | [02 Conceptos básicos de *NumPy*](https://github.com/PythonistaMX/py311/blob/main/02_conceptos_basicos_de_numpy.ipynb) · [03 Gestión de arreglos](https://github.com/PythonistaMX/py311/blob/main/03_gestion_de_arreglos_de_numpy.ipynb) · [04 Arreglos con contenido aleatorio](https://github.com/PythonistaMX/py311/blob/main/04_arreglos_con_contenido_aleatorio.ipynb) · [05 Operaciones básicas con arreglos](https://github.com/PythonistaMX/py311/blob/main/05_operaciones_basicas_con_arreglos.ipynb) |
| *pandas* | [09 Introducción a *pandas*](https://github.com/PythonistaMX/py311/blob/main/09_introduccion_a_pandas.ipynb) · [10 Tipos de datos](https://github.com/PythonistaMX/py311/blob/main/10_tipos_de_datos_de_pandas.ipynb) · [11 Operaciones básicas con *dataframes*](https://github.com/PythonistaMX/py311/blob/main/11_operaciones_basicas_con_dataframes.ipynb) · [12 Índices y multiíndices](https://github.com/PythonistaMX/py311/blob/main/12_indices_y_multiindices.ipynb) · [13 Datos categóricos](https://github.com/PythonistaMX/py311/blob/main/13_datos_categoricos.ipynb) |

> ⚠️ **Nota: py311 usa *pandas* 2.** Sus *notebooks* se escribieron para *pandas* 2.x y este curso usa *pandas* 3, así que algunos resultados se verán distintos. Por ejemplo, allá el texto aparece con `dtype` `object` (aquí es `str`), se explica el parámetro `inplace` y la asignación encadenada como si modificaran el original (con *pandas* 3 ya no lo hacen sobre columnas o selecciones) y puede aparecer `SettingWithCopyWarning`, que ya no existe. La sección 1.1 de `03_pandas_3_esencial.ipynb` explica estos cambios. Cuando haya diferencias, se sigue lo que muestra este *notebook*.

### Preparación del entorno.

Se importarán las bibliotecas con sus alias convencionales, se ubicará la carpeta `data/` (que ya trae los CSV del curso, sin necesidad de internet) y se fijará la semilla `401` para que los resultados aleatorios sean reproducibles.

* La siguiente celda importará las bibliotecas y definirá `DATA` y `rng`.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)

* La siguiente celda mostrará las versiones instaladas. El curso usa *pandas* 3.

In [2]:
{"NumPy": np.__version__, "pandas": pd.__version__, "seaborn": sns.__version__}

{'NumPy': '2.5.4', 'pandas': '3.0.6', 'seaborn': '0.13.2'}

### Objetos de los *notebooks* anteriores.

Este *notebook* es independiente: la siguiente celda vuelve a crear lo que se necesita de los *notebooks* anteriores.

* La siguiente celda leerá `penguins.csv`, el *dataset* presentado en `03_pandas_3_esencial.ipynb`.

In [3]:
pinguinos = pd.read_csv(os.path.join(DATA, "penguins.csv"))
pinguinos.shape

(344, 7)

---
## 1. Tipos de variables.

Antes de resumir o graficar una variable hay que saber **de qué tipo es**, porque el tipo decide qué preguntas tienen sentido: no se promedia una especie ni se ordena una isla. En el flujo del EDA, clasificar las variables es el paso que conecta los datos crudos con todos los análisis posteriores, y su producto concreto es el **diccionario de datos** de la sección 1 de `06_diccionario_de_datos.ipynb`.

El tipo **estadístico** (lo que la variable representa) no es lo mismo que el `dtype` de *pandas* (cómo se guarda en memoria), pero conviene que coincidan. Los `dtype` y sus conversiones se detallan en [py311 · 10 Tipos de datos](https://github.com/PythonistaMX/py311/blob/main/10_tipos_de_datos_de_pandas.ipynb) y [py311 · 13 Datos categóricos](https://github.com/PythonistaMX/py311/blob/main/13_datos_categoricos.ipynb).

### 1.1 Taxonomía: cualitativas y cuantitativas.

**Variables cualitativas o categóricas:** sus valores son **etiquetas** que asignan cada observación a un grupo.

* **Nominales:** las categorías no tienen orden. Ejemplos: `species` (Adelie, Chinstrap, Gentoo), `island`, color de ojos.
  * **Dicotómicas o binarias:** caso particular con exactamente dos categorías. Ejemplos: `sex` (Female, Male), `survived` (0, 1), "fumador sí/no". Suelen codificarse como 0/1, lo que permite que su **media sea una proporción** (la media de `survived` es la tasa de supervivencia).
* **Ordinales:** las categorías tienen un **orden natural**, pero la distancia entre ellas no está definida. Ejemplos: clase del pasaje (Third < Second < First), nivel educativo, escala de satisfacción de 1 a 5. Que "Second" esté entre "Third" y "First" no significa que esté "a la mitad".

**Variables cuantitativas o numéricas:** sus valores son **cantidades** medidas o contadas, y las diferencias entre ellos tienen significado.

* **Discretas:** toman valores aislados, normalmente conteos enteros. Ejemplos: número de hermanos a bordo (`sibsp`), personas en la mesa (`size` en `tips`), huevos en un nido.
* **Continuas:** pueden tomar cualquier valor dentro de un intervalo; solo la precisión del instrumento las limita. Ejemplos: `body_mass_g`, `bill_length_mm`, la tarifa `fare`.

**Nota:** que una variable continua se **registre** como entero (`body_mass_g` se anotó redondeada a 25 o 50 g) no la vuelve discreta: el tipo depende de lo que se mide, no de cómo se anotó.

### 1.2 Escalas de medición de Stevens.

El psicólogo S. S. Stevens (1946) propuso clasificar las variables según **qué operaciones tienen sentido** con sus valores. Cada escala hereda las operaciones de la anterior y agrega una nueva:

| Escala | Qué permite | Operaciones válidas | Estadísticos válidos | Ejemplos |
|---|---|---|---|---|
| **Nominal** | Distinguir | $=$, $\neq$ | Frecuencias, proporciones, moda | Especie, isla, sexo |
| **Ordinal** | Ordenar | $=$, $\neq$, $<$, $>$ | Lo anterior + mediana, percentiles, rangos | Clase del pasaje, nivel educativo |
| **De intervalo** | Medir diferencias (el cero es arbitrario) | Lo anterior + $+$, $-$ | Lo anterior + media, desviación estándar, correlación | Temperatura en °C o °F, año calendario |
| **De razón** | Comparar proporciones (existe un **cero absoluto**) | Lo anterior + $\times$, $\div$ | Lo anterior + coeficiente de variación $CV = s / \bar{x}$, media geométrica | Masa, longitud, ingreso, edad, temperatura en kelvin |

Las escalas nominal y ordinal corresponden a las variables cualitativas; las de intervalo y razón, a las cuantitativas. La distinción entre intervalo y razón importa al hablar de **cocientes**: un pingüino de 6,000 g pesa el doble que uno de 3,000 g (razón), pero un día a 20 °C **no** es "el doble de caluroso" que uno a 10 °C (intervalo), porque 0 °C no significa "ausencia de temperatura".

**Ejemplo:** la siguiente celda comprueba que el cociente entre dos temperaturas depende de la unidad (escala de intervalo), mientras que el cociente entre dos masas no (escala de razón).

* La siguiente celda calculará el cociente de dos temperaturas en °C, °F y K, y el de dos masas en gramos y libras.

In [4]:
temperaturas_c = np.array([20.0, 10.0])
temperaturas_f = temperaturas_c * 9 / 5 + 32
temperaturas_k = temperaturas_c + 273.15
masas_g = np.array([6000.0, 3000.0])
masas_lb = masas_g / 453.592
{"°C": float(temperaturas_c[0] / temperaturas_c[1]), "°F": round(float(temperaturas_f[0] / temperaturas_f[1]), 2),
 "K": round(float(temperaturas_k[0] / temperaturas_k[1]), 3),
 "masa en g": float(masas_g[0] / masas_g[1]), "masa en lb": round(float(masas_lb[0] / masas_lb[1]), 3)}

{'°C': 2.0, '°F': 1.36, 'K': 1.035, 'masa en g': 2.0, 'masa en lb': 2.0}

**Interpretación:** en °C el cociente es 2, en °F es 1.36 y en kelvin apenas 1.035: "el doble" depende de dónde se puso el cero, así que no tiene significado. El cociente de las masas es 2 en cualquier unidad. **Error común que se evita:** decir "aumentó 50%" sobre una variable de intervalo (temperaturas en °C, años, puntajes estandarizados). Para esas variables se reportan **diferencias**, no porcentajes.

### 1.3 Otros tipos que aparecen en un EDA.

* **Fechas y tiempos:** un **instante** (fecha de muestreo, 2007-11-11) se comporta como escala de intervalo: se restan dos fechas, pero no se suman ni se dividen. Una **duración** (días transcurridos, segundos de 0 a 60 mph) sí es de razón. De una fecha se **derivan** variables útiles: año, mes, día de la semana, temporada.
* **Texto libre:** comentarios, descripciones o nombres (`name` en `mpg`). No se resumen con estadísticos; se usan para extraer variables (la marca de un auto) o requieren técnicas de procesamiento de texto.
* **Identificadores:** distinguen observaciones (`id_muestra`, número de boleto, código postal). Aunque parezcan números, **no son cantidades**: no se promedian ni se correlacionan. Su papel es servir de **llave** para detectar duplicados y unir tablas (`07_duplicados_e_inconsistencias.ipynb`).
* **Variables derivadas:** se calculan a partir de otras: `body_mass_kg`, `bill_ratio`, `tip_pct` = 100 · `tip` / `total_bill`, un puntaje *z*. Heredan la escala de sus ingredientes (un cociente de dos variables de razón es de razón) y conviene documentar la fórmula en el diccionario de datos.

### 1.4 Del tipo estadístico al `dtype` de *pandas* 3.

| Tipo estadístico | Escala | Resúmenes típicos | Gráficas típicas | `dtype` sugerido en *pandas* 3 |
|---|---|---|---|---|
| Nominal | Nominal | Frecuencias, moda | Barras | `category` (o `str` si las categorías no son fijas) |
| Dicotómica | Nominal | Proporción | Barras | `bool`, `boolean` (*nullable*) o `category` |
| Ordinal | Ordinal | Frecuencias, mediana, percentiles | Barras en orden | `category` **ordenada** (`pd.CategoricalDtype(..., ordered=True)`) |
| Discreta | Intervalo o razón | Media, mediana, frecuencias | Barras o histograma | `int64`; `Int64` (*nullable*) si tiene faltantes |
| Continua | Intervalo o razón | Media, mediana, desviación estándar, IQR | Histograma, caja, densidad | `float64` |
| Instante | Intervalo | Rango, conteos por periodo | Línea en el tiempo | `datetime64[us]` |
| Duración | Razón | Media, mediana | Histograma | `timedelta64` |
| Texto libre | — | Longitud, palabras frecuentes | — | `str` |
| Identificador | — | Ninguno (unicidad) | — | `str` |

**Ejemplo:** se clasificará cada columna de `penguins` y se comparará el `dtype` con el que *pandas* la leyó contra el sugerido. Este es el primer paso de la auditoría de cualquier *dataset*.

* La siguiente celda construirá la tabla `clasificacion` con el tipo estadístico, la escala, el `dtype` leído y el `dtype` sugerido de cada columna de `pinguinos`.

In [5]:
clasificacion = pd.DataFrame(
    [
        ("species", "cualitativa nominal", "nominal", "category"),
        ("island", "cualitativa nominal", "nominal", "category"),
        ("bill_length_mm", "cuantitativa continua", "razón", "float64"),
        ("bill_depth_mm", "cuantitativa continua", "razón", "float64"),
        ("flipper_length_mm", "cuantitativa continua (anotada en enteros)", "razón", "Int64"),
        ("body_mass_g", "cuantitativa continua (anotada en enteros)", "razón", "Int64"),
        ("sex", "cualitativa nominal dicotómica", "nominal", "category"),
    ],
    columns=["columna", "tipo_estadistico", "escala", "dtype_sugerido"],
).set_index("columna")
clasificacion.insert(2, "dtype_leido", pinguinos.dtypes.astype(str))
clasificacion

,tipo_estadistico,escala,dtype_leido,dtype_sugerido
columna,,,,
species,cualitativa nominal,nominal,str,category
island,cualitativa nominal,nominal,str,category
bill_length_mm,cuantitativa continua,razón,float64,float64
bill_depth_mm,cuantitativa continua,razón,float64,float64
flipper_length_mm,cuantitativa continua (anotada en enteros),razón,float64,Int64
body_mass_g,cuantitativa continua (anotada en enteros),razón,float64,Int64
sex,cualitativa nominal dicotómica,nominal,str,category


**Interpretación:** *pandas* acierta con las medidas continuas del pico (`float64`), pero lee las tres variables cualitativas como `str` (texto genérico, sin conjunto de valores válidos) y las dos medidas anotadas en enteros como `float64`, solo porque tienen faltantes. Ninguna de esas lecturas es un error grave, pero cada una tiene costo: con `str`, una errata (`"MALE "` con espacio) se convertiría en una categoría nueva sin aviso; con `float64`, una masa "3750.0" oculta que el dato se anotó en enteros. La decisión de convertir se toma en el ejercicio 1b de `06_diccionario_de_datos.ipynb`.

**Errores comunes que se evitan:** tratar un código numérico como cantidad (sección 1.8), olvidar el orden de una ordinal y confundir "anotado en enteros" con "discreto".

**💡 Ideas clave de 3.1 a 3.4:**
* Cualitativas: nominales (incluidas las dicotómicas) y ordinales. Cuantitativas: discretas y continuas.
* La escala (nominal, ordinal, de intervalo, de razón) dice qué operaciones y estadísticos son válidos; los cocientes solo tienen sentido en escala de razón.
* Fechas, texto libre, identificadores y variables derivadas también se clasifican y documentan.
* El `dtype` debe reflejar el tipo: `category` (ordenada si hace falta), `Int64`, `float64`, `datetime64[us]`, `str`.

**Advertencia:** son trampas comunes los códigos numéricos que en realidad son categorías (`pclass` = 1, 2, 3; códigos postales), los conteos guardados como `float` solo porque hay faltantes y los `bool` que en realidad son una categoría binaria.

### 1.5 `category`: nominal y ordinal.

En *pandas* 3 el texto ya se lee como `str`, que es eficiente. `category` sigue siendo útil para fijar el **conjunto de valores válidos** y, sobre todo, para variables **ordinales**.

**¿Por qué este paso?** Convertir a `category` declara explícitamente el **conjunto de valores válidos** de una variable cualitativa. Para una ordinal, además fija el **orden**, de modo que tablas, gráficas y comparaciones (`>=`) lo respeten.

* La siguiente celda convertirá `species` en categoría nominal y mostrará sus categorías.

In [6]:
tipos = pinguinos.copy()
tipos["species"] = tipos["species"].astype("category")
tipos["species"].cat.categories.tolist()

['Adelie', 'Chinstrap', 'Gentoo']

* La siguiente celda creará la variable ordinal `talla` a partir de la masa, solo para ilustrar.

In [7]:
tipos["talla"] = pd.cut(
    tipos["body_mass_g"],
    bins=[0, 3500, 4500, np.inf],
    labels=["chico", "mediano", "grande"],
    ordered=True,
)
tipos["talla"].value_counts(sort=False)

talla
chico       78
mediano    149
grande     115
Name: count, dtype: int64

**Interpretación:** `pd.cut` convierte una variable continua en ordinal con tres tallas; `value_counts(sort=False)` respeta el orden chico < mediano < grande en lugar de ordenar por frecuencia. **Ojo:** discretizar una variable continua **pierde información** (un pingüino de 3,501 g y otro de 4,499 g quedan en la misma talla); se hace para comunicar o para tablas, no para reemplazar la variable original.

* La siguiente celda comparará categorías ordenadas: cuenta las tallas de *mediano* en adelante.

In [8]:
(tipos["talla"] >= "mediano").sum()

np.int64(264)

* La siguiente celda comparará la memoria de `island` como `str` (respaldado por *PyArrow*) y como `category`.

In [9]:
{
    "str": pinguinos["island"].memory_usage(deep=True),
    "category": pinguinos["island"].astype("category").memory_usage(deep=True),
}

{'str': 4980, 'category': 521}

**Interpretación:** `category` usa unas 10 veces menos memoria que `str`, porque guarda cada isla una sola vez y, por fila, solo un código entero. Con 344 filas no importa; con millones, sí.

### 1.6 📖 Lectura opcional: tipos *nullable*, enteros con faltantes.

`flipper_length_mm` y `body_mass_g` son enteros en el origen, pero *pandas* los lee como `float64` porque tienen `NaN`. Los tipos *nullable* (`Int64`, `boolean`, con mayúscula) representan el faltante como `pd.NA` sin cambiar el tipo.

* La siguiente celda convertirá `flipper_length_mm` a `Int64`.

In [10]:
pinguinos["flipper_length_mm"].astype("Int64").head()

0     181
1     186
2     195
3    <NA>
4     193
Name: flipper_length_mm, dtype: Int64

### 1.7 📖 Lectura opcional: fechas.

`penguins` no trae fechas, así que se usará un pequeño registro ficticio de salidas de campo. En *pandas* 3, `pd.to_datetime` sobre texto produce por defecto `datetime64[us]` (microsegundos) en lugar de `datetime64[ns]`.

* La siguiente celda creará el registro con las fechas como texto.

In [11]:
salidas = pd.DataFrame({
    "fecha": ["2007-11-11", "2008-11-09", "2009-11-18", "2009-12-01"],
    "isla": ["Torgersen", "Biscoe", "Dream", "Biscoe"],
})
salidas.dtypes

fecha    str
isla     str
dtype: object

* La siguiente celda convertirá `fecha` a `datetime` y derivará la temporada, el día de la semana y los días transcurridos.

In [12]:
salidas["fecha"] = pd.to_datetime(salidas["fecha"])
salidas = salidas.assign(
    temporada=salidas["fecha"].dt.year,
    dia_semana=salidas["fecha"].dt.day_name(),
    dias_desde_inicio=(salidas["fecha"] - salidas["fecha"].min()).dt.days,
)
salidas, salidas["fecha"].dtype

(       fecha       isla  temporada dia_semana  dias_desde_inicio
 0 2007-11-11  Torgersen       2007     Sunday                  0
 1 2008-11-09     Biscoe       2008     Sunday                364
 2 2009-11-18      Dream       2009  Wednesday                738
 3 2009-12-01     Biscoe       2009    Tuesday                751,
 dtype('<M8[us]'))

### Ejercicio 1 · Clasifica las variables de `titanic`.

Clasifica por tipo estadístico estas columnas: `survived`, `pclass`, `sex`, `age`, `sibsp`, `fare`, `embarked`, `class`, `deck`, `alone`. Anota también qué `dtype` les asignó *pandas* y si lo cambiarías. Responde en la celda de respuesta (puedes usar una tabla).

**🎯 Objetivo:** aplicar la taxonomía y las escalas de Stevens a un *dataset* nuevo. **Qué se espera aprender:** a no fiarse del `dtype` leído para decidir el tipo estadístico, y a detectar códigos numéricos que en realidad son categorías.

* La siguiente celda leerá `titanic.csv` y mostrará sus tipos.

In [13]:
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
titanic.dtypes

survived         int64
pclass           int64
sex                str
age            float64
sibsp            int64
parch            int64
fare           float64
embarked           str
class              str
who                str
adult_male        bool
deck               str
embark_town        str
alive              str
alone             bool
dtype: object

**✅ Respuesta sugerida:**

| Columna | Tipo estadístico | `dtype` leído | `dtype` sugerido |
|---|---|---|---|
| `survived` | Categórica binaria (nominal) | `int64` | `category` o `bool` |
| `pclass` | Categórica **ordinal** (codificada con números) | `int64` | `category` ordenada |
| `sex` | Categórica nominal | `str` | `category` |
| `age` | Numérica continua | `float64` | `float64` |
| `sibsp` | Numérica discreta (conteo) | `int64` | `int64` |
| `fare` | Numérica continua | `float64` | `float64` |
| `embarked` | Categórica nominal | `str` | `category` |
| `class` | Categórica ordinal (First > Second > Third) | `str` | `category` ordenada |
| `deck` | Categórica (ordinal según la posición en el barco; muchos faltantes) | `str` | `category` |
| `alone` | Categórica binaria | `bool` | `bool` |

`pclass` y `class` son **la misma información**: hay redundancia.

### 1.8 Elegir el `dtype`: decisiones y consecuencias.

| Pregunta | Decisión | Si se elige mal |
|---|---|---|
| ¿El número **mide** algo o es un **código**? | Medida ⇒ numérico; código (`pclass`, códigos postales) ⇒ `category` o `str` | Se calculan medias y correlaciones sin sentido |
| ¿Es un **identificador**? | `str`, aunque parezca número | Se pierden ceros a la izquierda; se "promedian" identificadores |
| ¿El conjunto de valores válidos es **fijo**? | `category` con las categorías explícitas | Con `category`, un valor mal escrito se vuelve `NaN` **sin aviso**; con `str`, se cuela como categoría nueva |
| ¿Tiene **orden**? | `category` ordenada | Las tablas y gráficas salen en orden alfabético; no se puede comparar con `>=` |
| ¿Es un conteo con faltantes? | `Int64` (*nullable*) | Con `float64` aparecen valores como `2.0` y se oculta que es un conteo |

**Ejemplo 1: un código tratado como número.** `pclass` vale 1, 2 o 3. *pandas* la lee como `int64`, así que `describe` le calcula media y desviación estándar, aunque "la clase promedio 2.31" no significa nada. Como categoría, el resumen correcto es una tabla de frecuencias.

* La siguiente celda comparará el resumen de `pclass` como número y como categoría.

In [14]:
titanic["pclass"].describe().round(2), titanic["pclass"].astype("category").value_counts(sort=False)

(count    891.00
 mean       2.31
 std        0.84
 min        1.00
 25%        2.00
 50%        3.00
 75%        3.00
 max        3.00
 Name: pclass, dtype: float64,
 pclass
 1    216
 2    184
 3    491
 Name: count, dtype: int64)

**Ejemplo 2: un identificador leído como número.** Los códigos postales de la CDMX empiezan con 0. Si se leen como enteros, se pierde el cero y `01000` deja de coincidir con otros registros al unir tablas.

* La siguiente celda leerá tres códigos postales como número (por defecto) y como texto (`dtype=str`).

In [15]:
import io

texto_csv = "codigo_postal,alcaldia\n01000,Álvaro Obregón\n06700,Cuauhtémoc\n14000,Tlalpan\n"
(pd.read_csv(io.StringIO(texto_csv))["codigo_postal"].tolist(),
 pd.read_csv(io.StringIO(texto_csv), dtype={"codigo_postal": str})["codigo_postal"].tolist())

([1000, 6700, 14000], ['01000', '06700', '14000'])

**Ejemplo 3: `category` con categorías fijas.** Fijar las categorías protege contra valores inválidos, pero en *pandas* 3 un valor que no está entre las categorías se convierte en `NaN` y solo se emite un aviso de obsolescencia (`Pandas4Warning`); en una versión futura será un error. Por eso se **validan los valores antes** de convertir y se cuentan los faltantes antes y después.

* La siguiente celda buscará los valores inválidos y después convertirá a una categoría con valores fijos, capturando el aviso para mostrarlo.

In [16]:
especies_validas = ["Adelie", "Chinstrap", "Gentoo"]
capturada = pd.Series(["Adelie", "adelie", "Gentoo", "Chinstrap"])
invalidos = capturada[capturada.notna() & ~capturada.isin(especies_validas)].tolist()
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    convertida = capturada.astype(pd.CategoricalDtype(especies_validas))
invalidos, convertida.tolist(), int(convertida.isna().sum()), [type(a.message).__name__ for a in avisos]

(['adelie'], ['Adelie', nan, 'Gentoo', 'Chinstrap'], 1, ['Pandas4Warning'])

### Reto 1 · Ordinal en la práctica.

Convierte `titanic["class"]` en categoría **ordenada** de forma que `First` sea la **más alta**. Después calcula, con una comparación (`>=`), la proporción de pasajeros que viajaban en `Second` o superior, y la tasa de supervivencia por clase en el orden correcto.

**🎯 Objetivo:** construir una categoría ordenada y usarla en comparaciones y agrupaciones. **Qué se espera aprender:** que el orden declarado cambia el resultado de `>=` y el orden de las tablas.

* La siguiente celda muestra una solución posible.

In [17]:
orden_clase = pd.CategoricalDtype(["Third", "Second", "First"], ordered=True)
clase = titanic["class"].astype(orden_clase)
(clase >= "Second").mean().round(3), titanic["survived"].groupby(clase).mean().round(3)

(np.float64(0.449),
 class
 Third     0.242
 Second    0.473
 First     0.630
 Name: survived, dtype: float64)

**💡 Ideas clave de las secciones 1.5 a 1.8:**
* `category` fija los valores válidos y el orden; ahorra memoria.
* Un código numérico (clase, código postal) no es una cantidad; un identificador va como `str`.
* Fijar categorías puede convertir erratas en `NaN`: se validan los valores antes de convertir.

---
## ✅ Cierre.

**Práctica (en Classroom):** terminar los ejercicios pendientes de este *notebook* y, si es posible, un reto.

**Siguiente *notebook*:** `05_muestreo_y_sesgos_de_recoleccion.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>